In [1]:
import pyreadstat
import pandas as pd
# Load the Stata dataset
df, meta = pyreadstat.read_dta("ETFC81FLSR.DTA")

# Explore the data
print(df.head())  # first few rows

# Explore metadata
print(meta.column_names)        # variable names
print(meta.column_labels)       # variable labels (descriptions)
print(meta.value_labels)        # value labels (coded categories)

   inv_id v000  v001  v002  v003  v004    v005  v007  v008  sf008a  ...  \
0       1  ET8    10  1001     1     1  246239     4     1       1  ...   
1       2  ET8    10  1001     1     2       0     6     3       5  ...   
2       3  ET8    10  1001     1     3       0     7     3       5  ...   
3       4  ET8    10  1001     1     4  654431     8     3       1  ...   
4       5  ET8    10  1001     1     5  246239     4     1       1  ...   

   v2010c  v2010d  v2011a v2011b  v2011c v2011d v2012a v2012b v2012c v2012d  
0       7       0       8      2       7      0      7      0      6      0  
1     NaN     NaN     NaN    NaN     NaN    NaN    NaN    NaN    NaN    NaN  
2     NaN     NaN     NaN    NaN     NaN    NaN    NaN    NaN    NaN    NaN  
3       3       0       0      0       0      0      2      0      2      0  
4       4       0       8      1       7      0      7      0      6      0  

[5 rows x 2588 columns]
['inv_id', 'v000', 'v001', 'v002', 'v003', 'v004', 'v005

In [2]:
# Build a summary table
summary = pd.DataFrame({
    "Variable Name": meta.column_names,
    "Column Label": meta.column_labels,
    "Value Labels": [meta.variable_value_labels.get(var, None) for var in meta.column_names]
})

# Preview first 20 rows
print(summary.head(20))

# If you want to export to Excel/CSV for easier browsing:
summary.to_csv("ETFC81FLSR_metadata.csv", index=False)

   Variable Name                                       Column Label  \
0         inv_id                                       inventory id   
1           v000                                       survey phase   
2           v001                          region (country-specific)   
3           v002             district (country-specific named zone)   
4           v003                                        urban/rural   
5           v004                                    facility number   
6           v005                                      sample weight   
7           v007                   facility type (country-specific)   
8           v008              managing authority (country-specific)   
9         sf008a                   cs - facility operational status   
10          v009                                        interviewer   
11          v010                                        type of spa   
12         v010a                                   result of survey   
13    

In [3]:
# Distinct counts for v001 and v002
distinct_v001 = df["v001"].nunique()
distinct_v002 = df["v002"].nunique()

print("Distinct count in v001:", distinct_v001)
print("Distinct count in v002:", distinct_v002)

# If you want to see the actual unique values:
print("Unique values in v001:", df["v001"].unique())
print("Unique values in v002:", df["v002"].unique())

Distinct count in v001: 11
Distinct count in v002: 86
Unique values in v001: [10  2  3  6 11  8  9  4 12  7  5]
Unique values in v002: [1001 1002 1003 1004 1006 1007 1008 1009 1010 1099  201  202  203  204
  205  299  301  302  303  304  305  306  307  308  309  310  399  311
  602  603  604  606 1199 1101  801  802  803  804  899  808  999  901
  401  402  403  404  405  406  407  499  408  409  410  411  412  413
  414  415  416  417  418  419  420 1299 1201  701  799  702  703  708
  705  706  707  709  710  712  717  718  719  720  721  505  508  509
  599  501]


In [ ]:
#Summary — Ready method to compute three zonal aggregated percent indicators from the SPA facility file and prepare for merging with the measles linelist: compute zone-level percent = (facilities with service present and completed) ÷ (all sampled facilities in zone, excluding nonresponses) × 100; 
#use the columns v002, v004, v012b, v012c, v012d; follow the code below to produce clean, reproducible zonal variables and a merge key. 
#Source dataset: SPA facility dataset loaded in your notebook (ETFC81FLSR.DTA). Variables used: v002 (zone code), v004 (facility id), v012b (child immunization present & completed), v012c (sick children present & completed), v012d (growth monitoring present & completed).
#Value coding: 0 = no service, 1 = yes completed, 7 = not returned from field / treat as missing. Exclude 7 and NaN from denominators. 

In [5]:
import pandas as pd

# --- load your SPA dataframe as df (example uses pyreadstat earlier) ---
# df, meta = pyreadstat.read_dta("ETFC81FLSR.DTA")

# keep only relevant cols
cols = ['v002','v004','v012b','v012c','v012d']
spa = df[cols].copy()

# treat '7' as missing and ensure numeric
spa[['v012b','v012c','v012d']] = spa[['v012b','v012c','v012d']].replace({7: pd.NA})
spa['v002'] = pd.to_numeric(spa['v002'], errors='coerce')

# --- mapping v002 codes to zone names (use the labels you provided) ---
zone_map = {
  101: 'north western', 102: 'central', 103: 'eastern', 104: 'southern',
  105: 'western', 106: 'mekelle', 199: 'south eastern', 201: 'zone 1',
  202: 'zone 2', 203: 'zone 3', 204: 'zone 4', 205: 'zone 5', 299: 'unknown',
  301: 'gondar town / north gondar', 302: 'south gondar', 303: 'north wollo',
  304: 'south wollo', 305: 'north shewa', 306: 'east gojjam', 307: 'west gojjam',
  308: 'waghimera', 309: 'awi', 310: 'oromia special', 311: 'bahirdar town woreda',
  399: 'dessie town woreda / central gondar / west gondar', 401: 'west wellega',
  402: 'east wellega', 403: 'ilu aba bora', 404: 'jimma', 405: 'west shewa',
  406: 'north shewa / sululta town', 407: 'bishoft town / east shewa',
  408: 'arsi / asalla town', 409: 'west hararge', 410: 'east hararge',
  411: 'east bale', 412: 'borena', 413: 'sebeta town / south west shewa',
  414: 'guji', 415: 'adama special', 416: 'jimma special',
  417: 'shashemene town / west arsi', 418: 'kelem wellega', 419: 'horo gudru wellega',
  420: 'burayu town', 499: 'dukem special / lega tafo lega dadhi / ambo / bale / batu / buno bedele / finifi',
  501: 'dollo / erar / fafan / jarar / shebelle / sitti', 505: 'korahay', 508: 'afder',
  509: 'liben', 599: 'dawa / dollo / erar / fafan / jarar / nogob / shebelle / sitti / unknown',
  602: 'metekel', 603: 'assosa', 604: 'kamashi', 606: 'mao komo special',
  701: 'gurage', 702: 'hadiya', 703: 'kembata tembaro', 705: 'gedeo', 706: 'wolayita',
  707: 'south omo', 708: 'sheka', 709: 'kaffa', 710: 'gamo', 712: 'yem special',
  717: 'dawro', 718: 'basketo special', 719: 'konta special', 720: 'silte',
  721: 'halaba', 799: 'amaro special / bench sheko / burji  special / derashe special / goffa / konso /',
  801: 'agniwa', 802: 'nuer', 803: 'majang', 804: 'itang special', 808: 'gambella town',
  899: 'gambella town', 901: 'hakim / shenkor / harari', 999: 'abadir / aboker / amir nur / dire teyara / erer / hakim / unknown / jinela / she',
  1001: 'akaki kality sub city', 1002: 'nifas silk lafto sub city', 1003: 'kolfe / keranyo subcity',
  1004: 'gulele sub city', 1006: 'lideta sub city', 1007: 'arada sub city', 1008: 'addis ketema sub city',
  1009: 'yeka sub city', 1010: 'bole sub city', 1099: 'kirkos sub city / unknown',
  1101: 'cherkacherk factory / legehare operational / sabian', 1199: 'operationals: addis ketema, dire dawa, biyoawale, gende kore, jeldessa, legehare',
  1201: 'aleta chuko / hoko / shebedino', 1299: 'aleta chuko / aleta wondo / arbegona / aroressa / bensa / bilate zuria / bona zu'
}
# map to a new column
spa['zone_name'] = spa['v002'].map(zone_map).fillna('unknown')

# --- aggregation helper ---
def pct_by_zone(df, col):
    grp = df.groupby(['v002','zone_name'])[col].agg(
        n_yes = lambda x: (x==1).sum(),
        n_valid = lambda x: x.isin([0,1]).sum()
    ).reset_index()
    grp[f'{col}_pct'] = (grp['n_yes'] / grp['n_valid']) * 100
    grp.loc[grp['n_valid']==0, f'{col}_pct'] = pd.NA
    return grp[['v002','zone_name', f'{col}_pct', 'n_yes', 'n_valid']]

pct_immun = pct_by_zone(spa, 'v012b')
pct_sick   = pct_by_zone(spa, 'v012c')
pct_growth = pct_by_zone(spa, 'v012d')

# merge into one zonal table (preserves zone_name)
zonal = pct_immun.merge(pct_sick, on=['v002','zone_name'], how='outer', suffixes=('_immun','_sick')) \
                 .merge(pct_growth, on=['v002','zone_name'], how='outer')

# rename to requested variable names
zonal = zonal.rename(columns={
    'v012b_pct': 'aggregated_zonal_child_immunization_service_availability_pct',
    'v012c_pct': 'aggregated_zonal_sick_children_services_present_completed_pct',
    'v012d_pct': 'aggregated_zonal_growth_monitoring_services_present_completed_pct',
    'n_yes_immun': 'n_yes_immunization',
    'n_valid_immun': 'n_valid_immunization',
    'n_yes_sick': 'n_yes_sick_children',
    'n_valid_sick': 'n_valid_sick_children',
    'n_yes': 'n_yes_growth_monitoring',
    'n_valid': 'n_valid_growth_monitoring'
})

# flag small samples
zonal['small_sample_flag'] = zonal[['n_valid_immunization','n_valid_sick_children','n_valid_growth_monitoring']].min(axis=1).fillna(0) < 5

# reorder columns for clarity
cols_out = ['v002','zone_name',
            'aggregated_zonal_child_immunization_service_availability_pct','n_yes_immunization','n_valid_immunization',
            'aggregated_zonal_sick_children_services_present_completed_pct','n_yes_sick_children','n_valid_sick_children',
            'aggregated_zonal_growth_monitoring_services_present_completed_pct','n_yes_growth_monitoring','n_valid_growth_monitoring',
            'small_sample_flag']
zonal = zonal[cols_out]

# export
zonal.to_csv('zonal_service_availability_with_names.csv', index=False)
